In [ ]:
import sys
import os
sys.path
os.listdir()
os.chdir(os.getcwd().replace("\\","/").replace("/notebooks",""))
sys.path.append("src")

In [ ]:
from src.envConfig import EnvConfig
EnvConfig()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring, when, round as _round, sum as _sum

In [ ]:
# 1. Inicializa a sessão Spark
spark = SparkSession.builder \
    .appName("Histórico dividendos") \
    .getOrCreate()

In [ ]:
geral = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1") \
    .option("delimiter", ";") \
    .csv("asserts/fii_geral/*.csv")
)

In [ ]:
complemento = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1") \
    .option("delimiter", ";") \
    .csv("asserts/fii_complemento/*.csv")
)

In [ ]:
geral.show(truncate=False)

In [ ]:
df_calculado = (
    complemento
        .withColumn("Valor_Patrimonial_Cotas", col("Valor_Patrimonial_Cotas").cast("double"))
        .withColumn("Percentual_Dividend_Yield_Mes", col("Percentual_Dividend_Yield_Mes").cast("double"))
        .withColumn(
            "Dividendo_Dinheiro_Mes", 
            col("Valor_Patrimonial_Cotas") * (col("Percentual_Dividend_Yield_Mes"))
        )
    )

df_dividendo_pago = (
    df_calculado
    .groupBy(["CNPJ_Fundo_Classe","Data_Referencia"]) 
    .agg(
            _round(_sum("Dividendo_Dinheiro_Mes"), 4).alias("Dividendos_Cota"),
            _round(_sum("Percentual_Dividend_Yield_Mes"), 2).alias("DY_Acumulado_Percentual")
        )
    .orderBy(col("Dividendos_Cota").desc())
)


In [ ]:
df_inf_geral = (
    geral
    .withColumn("Ticker", substring(col("Codigo_ISIN"), 3, 4))
    .withColumn("Ticker", col("Ticker"))
)

In [ ]:
df_relatorio = (
    df_inf_geral
    .select(
        col("CNPJ_Fundo_Classe"),
        col("Ticker"),
        col("Nome_Fundo_Classe"),
        col("Segmento_Atuacao")
    )
).drop_duplicates()

In [ ]:
df_relatorio_final = (
    df_relatorio
    .join(df_dividendo_pago, on="CNPJ_Fundo_Classe", how="inner")
    .filter(col("Dividendos_Cota") != 0.0)
    .orderBy(
        col("Data_Referencia").desc(), 
        col("Ticker").asc()
    )
).drop_duplicates(subset=["Ticker", "Data_Referencia"])

In [ ]:
df_relatorio_final.filter(col("Ticker") == "HGLG").orderBy(col("Data_Referencia").asc()).show(truncate=False)

In [ ]:
dpa = (
    df_relatorio_final
    .groupby('CNPJ_Fundo_Classe', 'Ticker', 'Nome_Fundo_Classe', 'Segmento_Atuacao')
    .agg(
        _sum("Dividendos_Cota").alias("total_dividendo"),
        _sum(when(col("Dividendos_Cota") > 0, 1).otherwise(0)).alias("Meses_Com_Dividendo")
    )
    .withColumn("DPA", col("total_dividendo")/(col("Meses_Com_Dividendo")/12))
)

In [ ]:
dpa.filter(col("Ticker") == "SNEL").show(truncate=False)

In [ ]:
preco_teto = (
    dpa
    .withColumn(
        "Teto Projetivo 6%", (col("DPA")/0.06)
    )
    .withColumn(
        "Teto Projetivo 8%", (col("DPA")/0.08)
    )
    .withColumn(
        "Teto Projetivo 9%", (col("DPA")/0.09)
    )
    .withColumn(
        "Teto Projetivo 12%", (col("DPA")/0.12)
    )
)

In [ ]:
preco_teto.filter(col("Ticker") == "HGLG").show(truncate=False)

In [ ]:
preco_teto.filter(col("Ticker") == "GARE").show(truncate=False)